# Convert XML to ATLAS format

The ATLAS format is a simple TSV file with line numbers and text. I need to convert my TEI XML version of Calpurnius Siculus' first bucolic poem into ATLAS.

In [2]:
# Import lxml.etree.ElementTree as ET to work with XML data
import lxml.etree as ET

In [3]:
# Parse the XML file and create an ElementTree object
tree = ET.parse("../xml/phi0830.phi001.lat.xml")

In [4]:
# Preview the XML
print(ET.tostring(tree, pretty_print=True, encoding="unicode"))

<?xml-model href="http://www.tei-c.org/release/xml/tei/custom/schema/relaxng/tei_all.rng" type="application/xml" schematypens="http://relaxng.org/ns/structure/1.0"?>
<?xml-model href="http://www.tei-c.org/release/xml/tei/custom/schema/relaxng/tei_all.rng" type="application/xml"
	schematypens="http://purl.oclc.org/dsdl/schematron"?>
<TEI xmlns="http://www.tei-c.org/ns/1.0">
   <teiHeader>
      <fileDesc>
         <titleStmt>
            <title>Title</title>
         </titleStmt>
         <publicationStmt>
            <p>Publication Information</p>
         </publicationStmt>
         <sourceDesc>
            <p>Information about the source</p>
         </sourceDesc>
      </fileDesc>
   </teiHeader>
   <text>
      <body>
         <sp who="Corydon">
            <speaker>C.</speaker>
            <l n="1">Nondum Solis equos declinis mitigat aestas,</l>
            <l n="2">quamvis et madidis incumbant prela racemis</l>
            <l n="3">et spument rauco ferventia musta susurro.</l>
  

In [5]:
# Register a namespace for the TEI elements
ns = {"tei": "http://www.tei-c.org/ns/1.0"}

I'm going to process the XML line by line and simultaneously print it to the screen and write it to a new TSV file. The XML is divided into sections by speaker (`<sp>`). I'm going to preserve the speaker labels inline because they are part of the textual tradition. Yes, they mess with the meter, but meter is a separate concern that can be handled under a different process.

There might be a better way of doing this, but I'm going to loop over the lines in each speaker section and make a list of tuples containing line number, line. When I loop over that list, if the item is the first one (index 0), I'll add the speaker's label after the number and before the text. Otherwise, I'll just get the number and the line.

In [6]:
# Open a TSV file for writing
with open("output.tsv", "w") as f:
    # Iterate through each <sp> element in the XML tree
    for sp in tree.xpath("//tei:sp", namespaces=ns):
        # Get the speaker's name from the "who" attribute and convert it to uppercase
        speaker = sp.xpath(".//tei:speaker", namespaces=ns)[0].text.upper()
        # Initialize an empty list to hold the tuples of line numbers and content
        content = []
        for l in sp.xpath(".//tei:l", namespaces=ns):
            n = l.get("n")
            line = l.text
            content.append((n, line))
        for item in content:
            n = item[0]
            line = item[1]
            # If the item's index is 0, print the speaker's name before the line content
            if content.index(item) == 0:
                print(f"{n}\t{speaker} {line}")
                f.buffer.write(f"{n}\t{speaker} {line}\n".encode("utf-8"))
            else:
                print(f"{n}\t{line}")
                f.buffer.write(f"{n}\t{line}\n".encode("utf-8"))

1	C. Nondum Solis equos declinis mitigat aestas,
2	quamvis et madidis incumbant prela racemis
3	et spument rauco ferventia musta susurro.
4	cernis ut ecce pater quas tradidit, Ornyte, vaccae
5	molle sub hirsuta latus explicuere genista?
6	nos quoque vicinis cur non succedimus umbris?
7	torrida cur solo defendimus ora galero?
8	O. Hoc potius, frater Corydon, nemus, antra petamus
9	ista patris Fauni, graciles ubi pinea denset
10	silva comas rapidoque caput levat obvia soli,
11	bullantes ubi fagus aquas radice sub ipsa
12	protegit et ramis errantibus implicat umbras.
13	C. Quo me cumque vocas, sequor, Ornyte; nam mea Leuce,
14	dum negat amplexus nocturnaque gaudia nobis,
15	pervia cornigeri fecit sacraria Fauni.
16	 prome igitur calamos et si qua recondita servas;
17	nec tibi defuerit mea fistula, quam mihi nuper
18	matura docilis compegit arundine Ladon.
19	O. Et iam captatae pariter successimus umbrae.
20	sed quaenam sacra descripta est pagina fago,
21	quam modo nescio quis properanti f

For future reference, there is a more elegant way to do this. ChatGPT analyzed my code and suggested this instead:

```python
with open("test.tsv", "w", encoding="utf-8") as f:
    for sp in tree.xpath("//tei:sp", namespaces=ns):
        speaker = sp.xpath("./tei:speaker", namespaces=ns)[0].text.upper()

        for i, line in enumerate(sp.xpath("./tei:l", namespaces=ns)):
            n = line.get("n")
            text = line.text

            if i == 0:
                text = f"{speaker} {text}"

            row = f"{n}\t{text}"
            print(row)
            f.write(f"{row}\n")
```

I tested it, and it did the job well. But my way was good enough for this purpose.